# PhishGuard AI — Baseline ML Model Training Notebook

This notebook demonstrates the baseline machine learning pipeline used by PhishGuard AI.
It is for exploration and documentation only — the production model is trained and served
by `backend/app/services/ml_classifier.py`.

**All email samples are synthetic. No real user data was used.**

In [ ]:
import sys
sys.path.insert(0, '../backend')

from app.services.ml_classifier import SYNTHETIC_DATA, LABELS
from collections import Counter

texts = [t for t, _ in SYNTHETIC_DATA]
labels = [l for _, l in SYNTHETIC_DATA]

print(f'Total samples: {len(texts)}')
print('Class distribution:', Counter(labels))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=10_000, sublinear_tf=True)),
    ('clf', CalibratedClassifierCV(LinearSVC(max_iter=2000, C=1.0)))
])

# Cross-validation on the synthetic corpus
scores = cross_val_score(pipeline, texts, labels, cv=5, scoring='accuracy')
print(f'5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred, target_names=sorted(set(labels))))

## Inference Example

Demonstrate classification on new synthetic examples.

In [ ]:
import numpy as np

test_emails = [
    "Your account has been suspended. Click here to verify: http://paypa1-secure.evil.com",
    "Are you available? I need you to process a wire transfer urgently. Keep this confidential.",
    "Congratulations! You've won a $500 Amazon gift card. Claim now!",
    "Hi team, the quarterly report is attached. Please review before Friday's meeting.",
]

proba = pipeline.predict_proba(test_emails)
classes = pipeline.classes_

for email, prob in zip(test_emails, proba):
    pred_idx = np.argmax(prob)
    print(f"Email: {email[:60]}...")
    print(f"  Prediction: {classes[pred_idx]} ({prob[pred_idx]:.2%} confidence)")
    print()

## Notes on the Baseline Model

- **Training data:** 70 synthetic labeled samples (20 phishing, 15 BEC, 15 spam, 20 legitimate)
- **This is a baseline model only.** Production deployment would require a much larger,
  vetted dataset.
- **The ML model is a supporting signal**, not the primary decision-maker. The rule-based
  engine drives the final classification.
- **Future improvements:** larger corpus, BERT-based embeddings, active learning loop,
  online threat intel feeds.